# Q2: Data Cleaning

**Phase 3:** Data Cleaning & Preprocessing  
**Points: 9 points**

**Focus:** Handle missing data, outliers, validate data types, remove duplicates.

**Lecture Reference:** Lecture 11, Notebook 1 ([`11/demo/01_setup_exploration_cleaning.ipynb`](https://github.com/christopherseaman/datasci_217/blob/main/11/demo/01_setup_exploration_cleaning.ipynb)), Phase 3. Also see Lecture 05 (data cleaning).

---

## Setup

In [7]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load data from Q1 (or directly from source)
df = pd.read_csv('data/beach_sensors.csv')
# If you saved cleaned data from Q1, you can load it:
# df = pd.read_csv('output/q1_exploration.csv')  # This won't work - load original

print("Libraries imported and data loaded.")

Libraries imported and data loaded.


---

## Objective

Clean the dataset by handling missing data, outliers, validating data types, and removing duplicates.

**Time Series Note:** For time series data, forward-fill (`ffill()`) is often appropriate for missing values since sensor readings are continuous. However, you may choose other strategies based on your analysis.

---

## Required Artifacts

You must create exactly these 3 files in the `output/` directory:

### 1. `output/q2_cleaned_data.csv`
**Format:** CSV file
**Content:** Cleaned dataset with same structure as original (same columns)
**Requirements:**
- Same columns as original dataset
- Missing values handled (filled, dropped, or imputed)
- Outliers handled (removed, capped, or transformed)
- Data types validated and converted
- Duplicates removed
- **Sanity check:** Dataset should retain most rows after cleaning (at least 1,000 rows). If you're removing more than 50% of data, reconsider your strategy—imputation is usually preferable to dropping rows for this dataset.
- **No index column** (save with `index=False`)

### 2. `output/q2_cleaning_report.txt`
**Format:** Plain text file
**Content:** Detailed report of cleaning operations
**Required information:**
- Rows before cleaning: [number]
- Missing data handling method: [description]
  - Which columns had missing data
  - Method used (drop, forward-fill, impute, etc.)
  - Number of values handled
- Outlier handling: [description]
  - Detection method (IQR, z-scores, domain knowledge)
  - Which columns had outliers
  - Method used (remove, cap, transform)
  - Number of outliers handled
- Duplicates removed: [number]
- Data type conversions: [list any conversions]
- Rows after cleaning: [number]

**Example format:**
```
DATA CLEANING REPORT
====================

Rows before cleaning: 50000

Missing Data Handling:
- Water Temperature: 2500 missing values (5.0%)
  Method: Forward-fill (time series appropriate)
  Result: All missing values filled
  
- Air Temperature: 1500 missing values (3.0%)
  Method: Forward-fill, then median imputation for remaining
  Result: All missing values filled

Outlier Handling:
- Water Temperature: Detected 500 outliers using IQR method (3×IQR)
  Method: Capped at bounds [Q1 - 3×IQR, Q3 + 3×IQR]
  Bounds: [-5.2, 35.8]
  Result: 500 values capped

Duplicates Removed: 0

Data Type Conversions:
- Measurement Timestamp: Converted to datetime64[ns]

Rows after cleaning: 50000
```

### 3. `output/q2_rows_cleaned.txt`
**Format:** Plain text file
**Content:** Single integer number (total rows after cleaning)
**Requirements:**
- Only the number, no text, no labels
- No whitespace before or after
- Example: `50000`

---

## Requirements Checklist

- [ ] Missing data handling strategy chosen and implemented
- [ ] Outliers detected and handled (IQR method, z-scores, or domain knowledge)
- [ ] Data types validated and converted
- [ ] Duplicates identified and removed
- [ ] Cleaning decisions documented in report
- [ ] All 3 required artifacts saved with exact filenames

---

## Your Approach

1. **Handle missing data** - Choose appropriate strategy (drop, forward-fill, impute) based on data characteristics
2. **Detect and handle outliers** - Use IQR method or z-scores; decide whether to remove, cap, or transform
3. **Validate data types** - Ensure numeric and datetime columns are properly typed
4. **Remove duplicates**
5. **Document and save** - Write detailed cleaning report explaining your decisions

---

## Decision Points

- **Missing data:** Should you drop rows, impute values, or forward-fill? Consider: How much data is missing? Is it random or systematic? For time series, forward-fill is often appropriate.
- **Outliers:** Are they errors or valid extreme values? Use IQR method or z-scores to detect, then decide: remove, cap, or transform. Document your reasoning.
- **Data types:** Are numeric columns actually numeric? Are datetime columns properly formatted? Convert as needed.

---

## Checkpoint

After Q2, you should have:
- [ ] Missing data handled
- [ ] Outliers addressed
- [ ] Data types validated
- [ ] Duplicates removed
- [ ] All 3 artifacts saved: `q2_cleaned_data.csv`, `q2_cleaning_report.txt`, `q2_rows_cleaned.txt`

---

**Next:** Continue to `q3_data_wrangling.md` for Data Wrangling.


In [8]:
# 1. Make a cleaned data csv

# Track imputation statistics for best data preservation
imputation_stats = {
    'rows_with_any_imputation': set(),
    'column_imputations': {}
}

print("="*70)
print("Data Cleaning and Preparation")
print("="*70)

# Convert 'timestamp' to datetime
print("1a. Convert 'Measurement Timestamp' to datetime format")
df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'], errors ='coerce')
invalid_timestampes = df['Measurement Timestamp'].isna().sum()
print(f"  - Found {invalid_timestampes} invalid timestamps which were set to NaT")

# Remove rows with invalid timestamps
if invalid_timestampes > 0:
    df = df.dropna(subset=['Measurement Timestamp'])
    print(f"  - Dropped {invalid_timestampes} rows with invalid timestamps")
    print(f"Timestamp conversion complete. Rows ramining: {len(df)}")

# Remove duplicate rows
print("\n1b. Remove Duplicate Rows")
initial_rows = len(df)
df = df.drop_duplicates()
duplicates_removed = initial_rows - len(df)
print(f"  - Removed {duplicates_removed}")
print(f"Rows Remaining: {len(df)}")

# Sort by timestamp
print("\n1c. Sort Data by Timestamp")
df = df.sort_values(by='Measurement Timestamp').reset_index(drop=True)
print("Data sorted by 'Measurement Timestamp'.")

# Handle numeric columns & track imputations
print("\n1d. Process Numeric Columns")
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(f"  - Numeric columns identified: {numeric_cols}")
print(f"{len(numeric_cols)} numeric columns found.")

for col in numeric_cols:
    missing_mask = df[col].isnull()
    missing_before = missing_mask.sum()

    if missing_before > 0:
        missing_pct = (missing_before / len(df)) * 100
        print(f"\n Processing'{col}':")
        print(f"  - Missing values: {missing_before} ({missing_pct:.2f}%)")

        # Get indices of rows with missing values that will be imputed
        rows_to_impute = df[missing_mask].index.tolist()

        # Use forward fill then backward fill for time series data to impute missing values. This is better than mean/median for time series.
        df[col] = df[col].ffill().bfill()

        # If there are still missing values (e.g., at start/end), fill with column median
        if df[col].isnull().sum() > 0:
            median_value = df[col].median()
            df[col] = df[col].fillna(median_value)
            print(f"  - Filled remaining missing values with median: {median_value:.2f}")
        else:
            print("  - Filled using forward/backward fill")

        # Track imputation
        imputation_stats['column_imputations'][col] = len(rows_to_impute)
        imputation_stats['rows_with_any_imputation'].update(rows_to_impute)


        missing_after = df[col].isnull().sum()
        print(f"  - Missing values after processing: {missing_after}")
        print(f"  - Total rows imputed: {len(rows_to_impute)}")

print(f"\n Numeric columns processed.")

# Handle Outliers (cap extreme values instead of removing rows)
print("\n1e. Handle Outliers in Numeric Columns")
outliers_handled = {}

for col in numeric_cols:
    if df[col].notna().sum() > 0:
        # Use IQR method to identify outliers
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Identify outliers
        outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_count = outlier_mask.sum()

        if outlier_count > 0:
            # Cap outliers instead of removing (to preserve data)
            df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
            outliers_handled[col] = outlier_count
            print(f"  - '{col}': Capped {outlier_count} outliers")

            # Track rows with outlier capping
            imputation_stats['rows_with_any_imputation'].update(df[outlier_mask].index.tolist())

total_outliers = sum(outliers_handled.values())
print(f"Total outliers capped: {total_outliers}")

# Validate categorical/text columns
print("\nf. Validate Categorical/Text Columns")
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
# Remove timestamp column from categorical list
categorical_cols = [col for col in categorical_cols if col != 'Measurement Timestamp']

for col in categorical_cols:
    missing_mask = df[col].isnull()
    missing_before = missing_mask.sum()
    
    if missing_before > 0:
        # Track rows being imputed
        rows_to_impute = df[missing_mask].index.tolist()
        imputation_stats['column_imputations'][col] = len(rows_to_impute)
        imputation_stats['rows_with_any_imputation'].update(rows_to_impute)

        # Fill missing categorical values with 'Unknown'
        df[col] = df[col].fillna('Unknown')
        print(f"  - '{col}': Filled {missing_before} missing values with 'Unknown'")
        print(f"  - Rows imputed: {len(rows_to_impute)}")

print(f"\n Categorical columns processed.")

# Imputation Summary
print("\n" + "=" * 70)
print("Imputation Summary")
print("=" * 70)

total_imputed_rows = len(imputation_stats['rows_with_any_imputation'])
imputation_rate = (total_imputed_rows / len(df)) * 100

print(f"Total rows with ANY imputation/modification: {total_imputed_rows:,} ({imputation_rate:.2f}%)")
print(f"Total rows unchanged: {len(df) - total_imputed_rows:,} ({100 - imputation_rate:.2f}%)")

if imputation_stats['column_imputations']:
    print("\nImputations by Column:")
    for col, count in sorted(imputation_stats['column_imputations'].items(),
                             key=lambda x: x[1], reverse=True):
        pct = (count/len(df)) * 100
        print(f"  -{col}: {count:,} values imputed ({pct:.2f}%)")

if outliers_handled:
    print("\nOutliers capped by column:")
    for col, count in sorted(outliers_handled.items(), key =lambda x: x[1], reverse=True):
        pct = (count/len(df)) * 100
        print(f"  -{col}: {count:,} outliers capped ({pct:.2f}%)")


# Final Validation!!!
print("\n" + "=" * 70)
print("Data Cleaning Summary")
print("=" * 70)
print(f"Original Rows: {initial_rows}")
print(f"Final rows after cleaning: {len(df)}")
print(f"Rows removed: {initial_rows - len(df)} ({((initial_rows - len(df)) / initial_rows) * 100:.2f}%)")
print(f"Columns: {df.shape[1]}")
print(f"\nDate range: {df['Measurement Timestamp'].min()} to {df['Measurement Timestamp'].max()}")
print(f"Time span: {(df['Measurement Timestamp'].max() - df['Measurement Timestamp'].min()).days} days")

# Check for remaining missing vlaues
remaining_missing = df.isnull().sum().sum()
print(f"\nRemaining missing values in dataset: {remaining_missing}")

if remaining_missing > 0:
    print("\nColumns with missing values:")
    for col in df.columns:
        missing = df[col].isnull().sum()
        if missing >0:
            print(f"  - '{col}': {missing} missing values")

# Sanity Check
if len(df) < 1000:
    print("\nWARNING: Less than 1000 rows remaining after cleaning. Please re-evaluate cleaning steps.")
else:
    print(f"\nPASSED: Dataset retains {len(df):,} rows (well above 1000 threshold)")

retention_rate = (len(df) / initial_rows) * 100
if retention_rate < 50:
    print(f"WARNING: Retained only ({retention_rate:.2f}%) of original data. Please re-evaluate cleaning steps.")
else:
    print(f"PASSED: Retained ({retention_rate:.2f}%) of original data.")


# Save cleaned data
print("\n" + "=" * 70)
print("Saving Cleaned Data")
print("=" * 70)

df.to_csv('output/q2_cleaned_data.csv', index=False)
print("Cleaned data saved to 'output/q2_cleaned_data.csv'")
print(f"    Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\n Data cleaning and preparation complete.")

# Display first few rows of cleaned data
print("\n" + "=" * 70)
print("Preview of Cleaned Data")
print("=" * 70)
print(df.head(10).to_string())
print("\n" + df.tail(5).to_string())



Data Cleaning and Preparation
1a. Convert 'Measurement Timestamp' to datetime format
  - Found 0 invalid timestamps which were set to NaT

1b. Remove Duplicate Rows
  - Removed 0
Rows Remaining: 196056

1c. Sort Data by Timestamp
Data sorted by 'Measurement Timestamp'.

1d. Process Numeric Columns
  - Numeric columns identified: ['Air Temperature', 'Wet Bulb Temperature', 'Humidity', 'Rain Intensity', 'Interval Rain', 'Total Rain', 'Precipitation Type', 'Wind Direction', 'Wind Speed', 'Maximum Wind Speed', 'Barometric Pressure', 'Solar Radiation', 'Heading', 'Battery Life']
14 numeric columns found.

 Processing'Air Temperature':
  - Missing values: 75 (0.04%)
  - Filled using forward/backward fill
  - Missing values after processing: 0
  - Total rows imputed: 75

 Processing'Wet Bulb Temperature':
  - Missing values: 75818 (38.67%)
  - Filled using forward/backward fill
  - Missing values after processing: 0
  - Total rows imputed: 75818

 Processing'Rain Intensity':
  - Missing value

In [9]:
# 2. Make an output file 'output/q2_cleaning_report.txt'
print("="*80)
print("Generating Cleaning Report")
print("="*80)

# INITIAL CLEANUP

# To calculate rows before cleaning, reload original data
df_original = pd.read_csv('data/beach_sensors.csv')
initial_rows = len(df_original)

# Load cleaned data from previous question
df_cleaned = pd.read_csv('output/q2_data_cleaned.csv')
final_rows = len(df_cleaned)

# Convert timestamp in original dataset for data analysis
df_original['Measurement Timestamp'] = pd.to_datetime(df_original['Measurement Timestamp'], errors='coerce')

# Prep report content
report_lines = []

def add_line(text="", indent=0):
    """To add formatted lines to the report."""
    report_lines.append(" " * indent + text)

def add_section_header(title):
    """To add section headers to the report with a separator."""
    add_line()
    add_line(title)
    add_line("="*len(title))
    add_line()

# Report Title
add_line("DATA CLEANING REPORT")
add_line("="*80)
add_line()
add_line(f"Dataset: Chicago Beach Sensor Data")
add_line()

# Overall Summary 
add_section_header("Overall Data Cleaning Summary")
add_line(f"Rows Before Cleaning: {initial_rows:,}")
add_line(f"Rows After Cleaning: {final_rows:,}")
rows_removed = initial_rows - final_rows
add_line(f"Rows Removed: {rows_removed:,} ({(rows_removed / initial_rows) * 100:.2f}%)")
add_line(f"Rows Retained: {final_rows:,} ({(final_rows / initial_rows) * 100:.2f}%)")
add_line(f"Columns: {df_cleaned.shape[1]}")

# Data Type Conversions
add_section_header("Data Type Conversions")
add_line(" - 'Measurement Timestamp' converted to datetime format.")

# Check for invalid timestamps
invalid_timestamps = df_original['Measurement Timestamp'].isna().sum()
if invalid_timestamps > 0:
    add_line(f" -> Found {invalid_timestamps} invalid timestamps")
    add_line(f" -> Dropped {invalid_timestamps} rows.")
else:
    add_line(" -> No invalid timestamps found.")

# Duplicate Rows
add_section_header("Duplicate Removal")

# Count duplicates in original data vs. after dropping
df_temp = df_original.dropna(subset = ['Measurement Timestamp'])
duplicates_removed = len(df_temp) - len(df_temp.drop_duplicates())

if duplicates_removed > 0:
    add_line(f"Duplicates found: {duplicates_removed:,} ({duplicates_removed / len(df_temp) * 100:.2f}%)")
    add_line(f"Method: Removed duplicate rows, keeping the first occurrence.")
    add_line(f"Result: {duplicates_removed:,} duplicate rows removed.")
else:
    add_line("No duplicate rows found.")
    add_line("No duplicate rows detected")

# Data Sorting!!!
add_section_header("DATA SORTING")
add_line("Method: Sorted by 'Measurement Timestamp' in ascending order")
add_line("Result: Data chronologically ordered for time series analysis")

# Missing Data Handling
add_section_header("MISSING DATA HANDLING")

numeric_cols = df_original.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_original.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'Measurement Timestamp']  

Missing_found = False

# Check numeric columns
for col in numeric_cols:
    missing_before = df_original[col].isnull().sum()
    if missing_before > 0:
        missing_found = True
        missing_pct = (missing_before / len(df_original)) * 100
        
        add_line(f"Column: {col}")
        add_line(f"  Missing values: {missing_before:,} ({missing_pct:.2f}%)", indent=1)
        add_line(f"  Method: Forward-fill then backward-fill (time series appropriate)", indent=1)
        add_line(f"  Fallback: Median imputation for remaining values", indent=1)
        
        # Calculate median for reference
        median_val = df_original[col].median()
        add_line(f"  Median value: {median_val:.2f}", indent=1)
        add_line(f"  Result: All {missing_before:,} missing values filled", indent=1)
        add_line()

# Check categorical columns
for col in categorical_cols:
    missing_before = df_original[col].isnull().sum()
    if missing_before > 0:
        missing_found = True
        missing_pct = (missing_before / len(df_original)) * 100
        
        add_line(f"Column: {col}")
        add_line(f"  Missing values: {missing_before:,} ({missing_pct:.2f}%)", indent=1)
        add_line(f"  Method: Filled with 'forward/backward fill'", indent=1)
        add_line(f"  Result: All {missing_before:,} missing values filled", indent=1)
        add_line()

if not missing_found:
    add_line("No missing values detected in the dataset")

# OUTLIER HANDLING
add_section_header("OUTLIER HANDLING")
add_line("Detection Method: IQR (Interquartile Range) Method")
add_line(" Bounds: [Q1 - 1.5*IQR, Q3 + 1.5*IQR]")
add_line("Handling Method: Capped outliers to nearest bound instead of removal")
add_line()

outliers_found = False

for col in numeric_cols:
    if df_original[col].notna().sum() > 0:
        # Calculate IQR bounds
        q1 = df_original[col].quantile(0.25)
        q3 = df_original[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Count outliers
        outlier_mask = (df_original[col] < lower_bound) | (df_original[col] > upper_bound)
        outlier_count = outlier_mask.sum()
        
        if outlier_count > 0:
            outliers_found = True
            outlier_pct = (outlier_count / df_original[col].notna().sum()) * 100
            
            add_line(f"Column: {col}")
            add_line(f"  Outliers detected: {outlier_count:,} ({outlier_pct:.2f}%)", indent=1)
            add_line(f"  Q1: {q1:.2f}", indent=1)
            add_line(f"  Q3: {q3:.2f}", indent=1)
            add_line(f"  IQR: {iqr:.2f}", indent=1)
            add_line(f"  Lower bound: {lower_bound:.2f}", indent=1)
            add_line(f"  Upper bound: {upper_bound:.2f}", indent=1)
            add_line(f"  Method: Capped values to bounds", indent=1)
            add_line(f"  Result: {outlier_count:,} values capped", indent=1)
            add_line()

if not outliers_found:
    add_line("No outliers detected in the dataset")

# DATA QUALITY SUMMARY
add_section_header("DATA QUALITY SUMMARY")

# Check final data quality
df_cleaned['Measurement Timestamp'] = pd.to_datetime(df_cleaned['Measurement Timestamp'])
df_cleaned_sorted = df_cleaned.sort_values('Measurement Timestamp')

add_line("Final Dataset Characteristics:")
add_line(f"  Total rows: {final_rows:,}", indent=1)
add_line(f"  Total columns: {df_cleaned.shape[1]}", indent=1)
add_line(f"  Date range: {df_cleaned_sorted['Measurement Timestamp'].min()} to {df_cleaned_sorted['Measurement Timestamp'].max()}", indent=1)
time_span = (df_cleaned_sorted['Measurement Timestamp'].max() - df_cleaned_sorted['Measurement Timestamp'].min()).days
add_line(f"  Time span: {time_span} days", indent=1)
add_line(f"  Remaining missing values: {df_cleaned.isnull().sum().sum()}", indent=1)
add_line(f"  Duplicate timestamps: {df_cleaned_sorted['Measurement Timestamp'].duplicated().sum()}", indent=1)
add_line()

# Data retention evaluation
add_line("Data Retention:")
retention_rate = (final_rows / initial_rows) * 100
if retention_rate >= 95:
    status = "EXCELLENT"
elif retention_rate >= 90:
    status = "GOOD"
elif retention_rate >= 80:
    status = "ACCEPTABLE"
else:
    status = "NEEDS REVIEW"

add_line(f"  Retention rate: {retention_rate:.2f}% ({status})", indent=1)
add_line(f"  Data loss: {(100-retention_rate):.2f}%", indent=1)


# DATA CLEANING RATIONALE
add_section_header("CLEANING RATIONALE")
add_line("Time Series Considerations:")
add_line("  - Forward-fill and backward-fill preserve temporal continuity", indent=1)
add_line("  - Suitable for sensor data where values change gradually", indent=1)
add_line("  - Median imputation used as fallback for robustness", indent=1)
add_line()
add_line("Outlier Treatment:")
add_line("  - Capping preserves data points while removing extreme values", indent=1)
add_line("  - Preferable to deletion for maintaining temporal continuity", indent=1)
add_line("  - IQR method robust to extreme outliers", indent=1)
add_line()
add_line("Data Preservation:")
add_line("  - Minimal row deletion (only invalid timestamps and duplicates)", indent=1)
add_line("  - Maximum retention of temporal information", indent=1)
add_line(f"  - Final dataset: {final_rows:,} rows retained", indent=1)

# Report footer
add_line()
add_line("="*80)
add_line("END OF REPORT")
add_line("="*80)


# Write report to file
report_content = "\n".join(report_lines)

with open('output/q2_cleaning_report.txt', 'w') as f:
    f.write(report_content)

print("\nCleaning report generated successfully!")
print(f"Saved to: output/q2_cleaning_report.txt")
print(f"Report length: {len(report_lines)} lines")

# Display report to console
print("\n" + "="*80)
print("REPORT PREVIEW")
print("="*80)
print(report_content)
print("\n" + "="*80)
print("Full report saved to output/q2_cleaning_report.txt")
print("="*80)

Generating Cleaning Report

Cleaning report generated successfully!
Saved to: output/q2_cleaning_report.txt
Report length: 260 lines

REPORT PREVIEW
DATA CLEANING REPORT

Dataset: Chicago Beach Sensor Data


Overall Data Cleaning Summary

Rows Before Cleaning: 196,056
Rows After Cleaning: 196,056
Rows Removed: 0 (0.00%)
Rows Retained: 196,056 (100.00%)
Columns: 18

Data Type Conversions

 - 'Measurement Timestamp' converted to datetime format.
 -> No invalid timestamps found.

Duplicate Removal

No duplicate rows found.
No duplicate rows detected

DATA SORTING

Method: Sorted by 'Measurement Timestamp' in ascending order
Result: Data chronologically ordered for time series analysis

MISSING DATA HANDLING

Column: Air Temperature
   Missing values: 75 (0.04%)
   Method: Forward-fill then backward-fill (time series appropriate)
   Fallback: Median imputation for remaining values
   Median value: 13.70
   Result: All 75 missing values filled

Column: Wet Bulb Temperature
   Missing values

In [10]:
print("="*80)
print("GENERATING q2_rows_cleaned.txt")
print("="*80)

# Load cleaned data
df_cleaned = pd.read_csv('output/q2_data_cleaned.csv')

# Get row count
row_count = len(df_cleaned)

print(f"\nCleaned dataset contains: {row_count:,} rows")

# Write to file - ONLY the number, no whitespace, no labels
with open('output/q2_rows_cleaned.txt', 'w') as f:
    f.write(str(row_count))

print(f"Saved to: output/q2_rows_cleaned.txt")
print(f"Content: {row_count}")

# Verify the file was written correctly
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

with open('output/q2_rows_cleaned.txt', 'r') as f:
    content = f.read()

print(f"File content: '{content}'")
print(f"Content length: {len(content)} characters")
print(f"Is pure integer: {content.isdigit()}")
print(f"No leading/trailing whitespace: {content == content.strip()}")

if content == str(row_count) and content.isdigit():
    print("\nFile format is CORRECT")
else:
    print("\nWARNING: File format may have issues")

print("\n" + "="*80)
print("COMPLETE")
print("="*80)

GENERATING q2_rows_cleaned.txt

Cleaned dataset contains: 196,056 rows
Saved to: output/q2_rows_cleaned.txt
Content: 196056

VERIFICATION
File content: '196056'
Content length: 6 characters
Is pure integer: True
No leading/trailing whitespace: True

File format is CORRECT

COMPLETE
